<p style="padding: 10px; border: 1px solid black;">
<img src="../common/images/mlu-logo.png" alt="drawing" width="400"/> <br/>
<div style="background-image: linear-gradient(145deg, rgba(35, 47, 62, 1) 0%, rgba(51, 0, 102, 1) 40%, rgba(223, 42, 93, 1) 60%, rgba(124, 90, 237, 1) 85%, rgba(124, 232, 244, 1) 100%); padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">MLU: Application of Deep Learning to Text and Image Data</h1>
    <h2 style="color: white; margin-top: 15px;">Implementing a CNN by Using PyTorch</h2>
</div>

<!-- Compact Lab Introduction with Activity/Challenge Explanation -->
<div style="background-color: #F8F9F9; padding: 15px; border-radius: 5px; margin: 20px 0;">
    <h4 style="color: #2E4053; margin-top: 0;">About This Lab</h4>
    <p>Throughout this lab, you will encounter two types of interactive elements:</p>
    <table style="width: 100%; border-collapse: collapse; margin: 15px 0;">
        <tr>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../common/images/mlu-activity.png" alt="Activity" width="125"/>
            </td>
            <td style="text-align: center; padding: 10px; width: 50%;">
                <img src="../common/images/mlu-challenge.png" alt="Challenge" width="125"/>
            </td>
        </tr>
        <tr>
            <td style="text-align: center; padding: 10px; background-color: #EBF5FB;">
                <p>No coding is needed for an activity. You try to understand a concept, <br/>answer questions, or run a code cell.</p>
            </td>
            <td style="text-align: center; padding: 10px; background-color: #FEF9E7;">
                <p>Challenges are where you test your understanding by implementing something new or taking a short quiz.</p>
            </td>
        </tr>
    </table>
    <p>Please work through this notebook from top to bottom to avoid errors due to missing code or context.</p>
</div>

<!-- Table of Contents with All Section Levels -->
<div style="background-color: #f2f0fc; padding: 15px; border-radius: 5px; margin-bottom: 30px;">
    <h2 style="color: #2f1381; border-bottom: 1px solid #2f1381; padding-bottom: 5px;">Table of Contents</h2>
    <p><a href="#section1" style="color: #2f1381; font-weight: bold; text-decoration: none;">1. Introduction of a real-world example</a></p>
    <p><a href="#section2" style="color: #2f1381; font-weight: bold; text-decoration: none;">2. Load the dataset</a></p>
    <p><a href="#section3" style="color: #2f1381; font-weight: bold; text-decoration: none;">3. Design the model architecture</a></p>
    <p><a href="#section4" style="color: #2f1381; font-weight: bold; text-decoration: none;">4. Define the loss function, optimizer, and evaluation metric</a></p>
    <p><a href="#section5" style="color: #2f1381; font-weight: bold; text-decoration: none;">5. Train the model</a></p>
    <p><a href="#section6" style="color: #2f1381; font-weight: bold; text-decoration: none;">6. Evaluate the model</a></p>
</div>

In [ ]:
# Remove conflicting packages that are not used by this notebook
!pip uninstall -y -q fastai autogluon-multimodal autogluon-timeseries torchtext timm 2>/dev/null || true
# Install libraries
!pip install -U -q -r requirements.txt

In [ ]:
import os
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torch.optim import SGD

<!-- Section Header -->
<div id="section1" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">1. Introduction of a real-world example</h2>
</div>

The [Materials in Context Database (MINC)](http://opensurfaces.cs.cornell.edu/publications/minc) from Cornell University is a large-scale dataset of images of real-world materials. This lab uses a subset of the MINC dataset. The dataset is well-labeled and a moderate size, which makes it a good fit for this example.

Reference: Sean Bell, Paul Upchurch, Noah Snavely, and Kavita Bala. "Material Recognition in the Wild with the Materials in Context Database." *Computer Vision and Pattern Recognition (CVPR)*, April 2015. https://arxiv.org/abs/1412.0623.

The following image provides examples of images from multiple classes of the dataset.

![MINC 2500 Examples by class](../common/images/MINC-2500.png)

<!-- Section Header -->
<div id="section2" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">2. Load the dataset</h2>
</div>

First, load the dataset that you will use to train the CNN model. In this example, you will use the MINC-2500 dataset with the following classes: 
- Brick
- Carpet
- Food
- Mirror
- Sky
- Water

To start, define the training, validation, and test paths.

In [ ]:
path = '../data/minc-2500'
train_path = os.path.join(path, 'train')
val_path = os.path.join(path, 'val')
test_path = os.path.join(path, 'test')

A good practice is to visualize the dataset. To do this, define the `show_images` function.

In [ ]:
def show_images(imgs, num_rows, num_cols, titles=None, scale=1.5):
    """Plot a list of images."""
    figsize = (num_cols * scale, num_rows * scale)
    _, axes = plt.subplots(num_rows, num_cols, figsize=figsize)
    axes = axes.flatten()
    for i, (ax, img) in enumerate(zip(axes, imgs)):
        ax.imshow(img.permute(1,2,0).numpy())
        ax.axes.get_xaxis().set_visible(False)
        ax.axes.get_yaxis().set_visible(False)
        if titles:
            ax.set_title(titles[i])
    return axes

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Visualize Sample Images</h4>
        <p>To load sample images from the test data, run the following cell. To see different images, run the cell multiple times.</p>
        <p>To display a 4x4 grid of images, update the call to the <code>show_images</code> function.</p>
    </div>
</div>

In [ ]:
test_dataset = ImageFolder(test_path, transform=transforms.ToTensor())
test_sample = DataLoader(test_dataset, batch_size=2*8, shuffle=True)

for data, label in test_sample:
    show_images(data, 2, 2);
    break

To load the dataset, you first need to prepare the image data by using `transfoms` functions as follows:
1. Load the image data and resize it to the given size (224,224).
1. Convert the image tensor of shape (C x H x W) in the range [0, 255] to a `float32` torch tensor of shape (C x H x W) in the range (0, 1) using the `ToTensor` class.
1. Normalize a tensor of shape (C x H x W) with its mean and standard deviation by using the `Normalize` function.

In [ ]:
transformation = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0,0,0), std=(1,1,1))
])

Now that you have defined a transformation, you can load the train, validation, and test sets and apply the transformation to them.

In practice, reading data can be a significant performance bottleneck, even when the model is simple or when the computer is fast. Loading the data can take more time than training the model. To speed up the process of loading the dataset, use a PyTorch `DataLoader`. A data loader reads a minibatch of data with size `batch_size` each time.

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Load Training and Validation Data</h4>
        <p>To load the train and validation sets, run the following cell.</p>
        <p>For a large dataset, you can adjust the <code>batch_size</code> to improve load speed.</p>
        <p><strong>Note:</strong> This dataset is small, and the Amazon SageMaker instance is fast, so adjusting the batch size here would have little impact on the data load.</p>
    </div>
</div>

In [ ]:
batch_size = 16

train_loader = DataLoader(
    ImageFolder(train_path, transform=transformation),
    batch_size=batch_size, shuffle=True)

validation_loader = DataLoader(
    ImageFolder(val_path, transform=transformation),
    batch_size=batch_size, shuffle=False)


<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Load Test Data</h4>
        <p>In the following cell, write code to load the test set.</p>
        <p><strong>Your task:</strong> Complete the code to create a DataLoader for the test dataset.</p>
    </div>
</div>

In [ ]:
############### CODE HERE ###############

test_loader = DataLoader(
    ImageFolder(test_path, transform=transformation),
    batch_size=batch_size, shuffle=False)

############## END OF CODE ##############

<!-- Section Header -->
<div id="section3" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">3. Design the model architecture</h2>
</div>

Now you will design a CNN. First, initialize a `Sequential` block. In PyTorch, `Sequential` defines a container for several layers that will be chained together. Given input data, a `Sequential` passes it through the first layer, in turn passing the output as the second layer's input and so forth.

You want to build a neural network with a 2D convolutional layer `Conv2D`, followed by a 2D max pooling layer `MaxPool2D`, a fully connected (or `Dense`) layer, and a final output `Dense` layer with six output classes. The following figure shows a diagram of the CNN architecture that you will build in this notebook.

<center><img src="../common/images/cnn.png" width="250px" alt="CNN architecture"></center>

In [ ]:
# Use GPU resource if available; otherwise, use CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

out_classes = 6

net = nn.Sequential(
    # First convolutional layer
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5),
    nn.ReLU(),
    # First max pooling layer
    nn.MaxPool2d(kernel_size=2, stride=2),
    # Second convolutional layer
    nn.Conv2d(in_channels=16, out_channels=16, kernel_size=5),
    nn.ReLU(),
    # Second max pooling layer
    nn.MaxPool2d(kernel_size=2, stride=2),
    # The flatten layer collapses all axes,
    # except the first one, into one axis.
    nn.Flatten(),
    # Fully connected or dense Layer
    nn.Linear(53*53*16, 32),
    nn.ReLU(),
    # Output layer
    nn.Linear(32, out_classes)).to(device)

<!-- Section Header -->
<div id="section4" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">4. Define the loss function, optimizer, and evaluation metric</h2>
</div>

The neural network is almost ready to be trained. The last thing to do before training is set the hyperparameters, such as training device (GPU or CPU), the number of epochs to train, and the learning rate of the optimization algorithm.

In [ ]:
epochs = 15
learning_rate = 0.01

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Specify Loss Function</h4>
        <p>To specify the loss function, run the following cell.</p>
        <p>Because this is a multiclass classification task, use <code>CrossEntropyLoss</code> as the loss function. Different problem types use different loss functions. For example, mean squared error (MSE) is commonly used for regression problems.</p>
    </div>
</div>

In [ ]:
criterion = nn.CrossEntropyLoss()

Now, you need to instantiate the `optim.<Optimizer>`, which defines the parameters to optimize over (which are obtained from the network by using `net.parameters()`) and the hyperparameters that the optimization algorithm requires.

In [ ]:
optimizer = SGD(net.parameters(), lr=learning_rate)

Finally, define a function to calculate the accuracy, `calculate_accuracy(output, label)`. This function is called for each batch of data. The function uses the network's outputs and the corresponding labels.

In [ ]:
def calculate_accuracy(output, label):
    """Calculate the accuracy of the trained network. 
    output: (batch_size, num_output) float32 tensor
    label: (batch_size, ) int32 tensor """
    
    return (output.argmax(axis=1) == label.float()).float().mean()

<!-- Section Header -->
<div id="section5" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">5. Train the model</h2>
</div>

<!-- Activity Box -->
<div style="background-color: #EBF5FB; border-left: 5px solid #3498DB; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-activity.png" alt="Activity" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #2874A6; margin-top: 0;">Activity: Train the Model</h4>
        <p>It's time to train the model! Run the following cell.</p>
        <p>This code will train 15 epochs (15 full passes through the dataset).</p>
    </div>
</div>

In [ ]:
for epoch in range(epochs):
    #You set the device in the "Design the model architecture" section
    net = net.to(device)

    train_loss, val_loss, train_acc, valid_acc = 0., 0., 0., 0.

    # Training loop
    # This loop trains the neural network (weights are updated)
    net.train() # Activate training mode
    for data, label in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()
        # Put data and label to the correct device
        data = data.to(device)
        label = label.to(device)
        # Make forward pass
        output = net(data)
        # Calculate loss
        loss = criterion(output, label)
        # Make backward pass (calculate gradients)
        loss.backward()
        # Accumulate training accuracy and loss
        train_acc += calculate_accuracy(output, label).item()
        train_loss += loss.item()
        # Update weights
        optimizer.step()

    # Validation loop
    # This loop tests the trained network on the validation dataset
    # No weight updates here
    # torch.no_grad() reduces memory usage when not training the network
    net.eval() # Activate evaluation mode
    with torch.no_grad():
        for data, label in validation_loader:
            data = data.to(device)
            label = label.to(device)
            # Make forward pass with the trained model so far
            output = net(data)
            # Accumulate validation accuracy and loss
            valid_acc += calculate_accuracy(output, label).item()
            val_loss += criterion(output, label).item()

    # Take averages
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    val_loss /= len(validation_loader)
    valid_acc /= len(validation_loader)

    print("Epoch %d: train loss %.3f, train acc %.3f, val loss %.3f, val acc %.3f" % (
        epoch+1, train_loss, train_acc, val_loss, valid_acc))

You might notice that the training loss and accuracy improve, while the validation loss and accuracy mostly fluctuate. This is a signal of overfitting.

<!-- Section Header -->
<div id="section6" style="border-left: 5px solid #2f1381; padding-left: 15px; margin: 40px 0 20px 0;">
    <h2 style="color: #2f1381;">6. Evaluate the model</h2>
</div>

<!-- Challenge Box -->
<div style="background-color: #FEF9E7; border-left: 5px solid #F1C40F; padding: 15px; border-radius: 5px; margin: 20px 0; display: flex; align-items: flex-start;">
    <div style="flex: 0 0 60px; margin-right: 15px;">
        <img src="../common/images/mlu-challenge.png" alt="Challenge" width="200" style="max-width: 100%; height: auto;">
    </div>
    <div style="flex: 1;">
        <h4 style="color: #B7950B; margin-top: 0;">Challenge: Evaluate Model Performance</h4>
        <p>In the following cell, write code that computes the test accuracy by using the evaluation function.</p>
        <p><strong>Your task:</strong> Complete the code to evaluate the model on the test dataset.</p>
    </div>
</div>

In [ ]:
test_acc = 0.
net.eval() # Activate evaluation mode
with torch.no_grad():
############### CODE HERE ###############
    for data, label in test_loader:
        data = data.to(device)
        label = label.to(device)
        output = net(data)
        test_acc += calculate_accuracy(output, label).item()
############## END OF CODE ##############

test_acc = test_acc/len(test_loader)

print("Test accuracy: %.3f" % test_acc)

Now that you have designed a CNN model and evaluated its accuracy, you are ready to build a model with a more modern architecture that performs better.

<div style="background-color: #f2f0fc; padding: 15px; border-radius: 5px; margin: 30px 0;">
    <h3 style="color: #2f1381; border-bottom: 1px solid #2f1381; padding-bottom: 5px;">Conclusion</h3>
    <p style="color: #2f1381;">In this lab, you have:</p>
    <ul>
        <li style="color: #2f1381;">Used built-in PyTorch CNN architectures to train a multiclass classification model</li>
        <li style="color: #2f1381;">Designed a CNN model architecture</li>
        <li style="color: #2f1381;">Trained a model by using a CNN</li>
        <li style="color: #2f1381;">Evaluated performance of a CNN model</li>
    </ul>
    <h4 style="color: #2f1381; margin-top: 15px;">Next Steps</h4>
    <p style="color: #2f1381;">In the next lab, you will learn how to build a model by using a modern architecture, ConvNeXt, with PyTorch.</p>
</div>

<p style="padding: 10px; border: 1px solid black;">
<img src="../common/images/mlu-logo.png" alt="drawing" width="400"/> <br/>

# Thank you!